# Estadística, tiempos de espera y reproducibilidad

Analiza conjuntos de trayectorias, distribuciones de tiempos de permanencia, variabilidad y semillas aleatorias.

> **Nota científica:** KineSIM es un prototipo de modelado cinético fenomenológico. Los parámetros iniciales son ilustrativos y los mapas de energía no están calibrados experimentalmente para kinesina.

## Resultados de aprendizaje

Este notebook estudia por qué una trayectoria individual no basta para caracterizar un motor molecular. Aprenderás a generar un conjunto de realizaciones, resumir distribuciones de velocidades y desplazamientos, comparar tiempos de permanencia con una ley exponencial y repetir una trayectoria mediante una semilla fija.

Las realizaciones son muestras del **modelo computacional** bajo las mismas condiciones. No deben confundirse con mediciones independientes de motores reales.

## Instalación desde PyPI

Este notebook utiliza la distribución publicada de **KineSIM**. No modifica `sys.path` ni importa código desde la carpeta `src/`.

- Nombre de distribución: `kinesim` (PyPI no distingue mayúsculas).
- Namespace de importación: `protein_motor_sim` (se conserva por compatibilidad).
- Versión fijada para reproducibilidad: `0.1.2`.

Ejecuta la siguiente celda con acceso a Internet. Si la versión todavía no aparece en PyPI, primero debe publicarse con Twine. Para probar una subida en TestPyPI, consulta las instrucciones de instalación de este README; no mezcles los índices de TestPyPI y PyPI en una misma resolución de dependencias.


In [ ]:
%pip install --quiet "kinesim[notebooks]==0.1.2"


## 1. Objetivos

Este notebook estudia la variabilidad entre motores simulados, los tiempos de espera y la reproducibilidad. Las realizaciones son independientes y usan semillas derivadas de una semilla maestra.

## Conceptos estadísticos

Una trayectoria de Gillespie es una realización posible de un proceso estocástico. Un conjunto de $N$ trayectorias permite estudiar la distribución de un observable $Y$:

$$
\overline{Y}=\frac{1}{N}\sum_{i=1}^{N}Y_i.
$$

La desviación estándar muestral cuantifica la dispersión entre realizaciones. En este notebook se muestran histogramas de velocidad media neta y desplazamiento neto. Los histogramas no representan incertidumbre experimental, sino la variabilidad que resulta de las tasas y del muestreo aleatorio del modelo.

In [ ]:
from importlib.metadata import version as distribution_version
print("KineSIM distribution:", distribution_version("kinesim"))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from protein_motor_sim import Environment, GillespieSimulator, Kinesin1
from protein_motor_sim.simulation.ensemble import run_ensemble
from protein_motor_sim.analysis.statistics import summarize_trajectories

## 2. Simular un conjunto de motores

Una trayectoria individual no caracteriza por sí sola el comportamiento estadístico. Ejecutamos muchas realizaciones bajo las mismas condiciones.

In [ ]:
motor = Kinesin1()
environment = Environment(atp_concentration=1e-3, external_force_N=0.0, temperature_K=310.0)
simulator = GillespieSimulator()
ensemble = run_ensemble(
    simulator, motor, environment, t_max=0.5, n_runs=200, seed=2026
)
summary = summarize_trajectories(ensemble.trajectories)
summary

### Interpretación del conjunto

Cada fila conceptual del conjunto es una trayectoria con su propia secuencia de eventos. La media resume el centro de la distribución y la desviación estándar resume su dispersión.

La velocidad de una trayectoria se calcula como desplazamiento neto dividido por la duración registrada. Si la kinesina se desprende antes de `t_max`, la observación termina en ese instante; por eso la duración efectiva puede variar entre realizaciones.

In [ ]:
velocities = np.array([tr.mean_velocity_nm_s for tr in ensemble.trajectories])
net_displacements = np.array([tr.displacement_nm for tr in ensemble.trajectories])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(velocities, bins=25, edgecolor="white")
axes[0].set_xlabel("Velocidad media neta (nm/s)")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución de velocidades")
axes[1].hist(net_displacements, bins=25, edgecolor="white")
axes[1].set_xlabel("Desplazamiento neto (nm)")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución de desplazamientos")
plt.tight_layout()
plt.show()

## 3. Distribución de tiempos de espera en un estado

Para una red de tasas constantes dentro de un estado dado, el tiempo de permanencia tiene distribución exponencial con parámetro igual a la suma de las tasas salientes. Extraemos los tiempos de eventos que salen de `bound_apo`. Se excluyen los intervalos censurados al final de observación.

### Modelo de tiempos de permanencia

Para un estado con tasas de salida constantes durante la permanencia, el tiempo hasta el siguiente evento tiene una distribución exponencial. Si las tasas salientes son $a_1,\ldots,a_M$, la tasa total es:

$$
a_0=\sum_{\mu=1}^{M}a_\mu.
$$

La densidad del tiempo de espera es:

$$
p(\tau)=a_0e^{-a_0\tau},\qquad \tau\geq0,
$$

con media $\langle\tau\rangle=1/a_0$. El notebook selecciona intervalos cuyo estado de origen es `bound_apo`. El evento que termina el intervalo puede ser unión de ATP o desprendimiento; se excluye el registro final `observation_end`, porque es un intervalo censurado por el límite de observación y no un evento de salida observado.

In [ ]:
dwell_times = []
for tr in ensemble.trajectories:
    for i in range(1, len(tr.times_s)):
        source_state = tr.states[i - 1]
        event = tr.events[i]
        if source_state == "bound_apo" and event != "observation_end":
            dwell_times.append(tr.times_s[i] - tr.times_s[i - 1])

dwell_times = np.asarray(dwell_times)
network = motor.build_network()
total_exit_rate = sum(rate for _, rate in network.rates("bound_apo", environment))
print(f"Intervalos observados: {len(dwell_times)}")
print(f"Tasa total teórica de salida: {total_exit_rate:.3f} s^-1")
print(f"Media teórica: {1 / total_exit_rate:.5f} s")
print(f"Media muestral: {dwell_times.mean():.5f} s")

In [ ]:
if len(dwell_times):
    tau = np.linspace(0, np.quantile(dwell_times, 0.99), 250)
    pdf = total_exit_rate * np.exp(-total_exit_rate * tau)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(dwell_times, bins=30, density=True, alpha=0.65, label="Simulación")
    ax.plot(tau, pdf, linewidth=2, label="Exponencial teórica")
    ax.set_xlabel("Tiempo de permanencia en bound_apo (s)")
    ax.set_ylabel("Densidad de probabilidad (s$^{-1}$)")
    ax.set_title("Distribución de tiempos de espera")
    ax.legend()
    ax.grid(alpha=0.2)
    plt.show()

### Cómo comparar teoría y simulación

El histograma se normaliza como densidad de probabilidad y se superpone la exponencial calculada con la tasa total de salida de `bound_apo` bajo el entorno elegido. La comparación comprueba la propiedad de espera exponencial del algoritmo cuando las tasas son constantes en ese estado.

La media muestral puede diferir de $1/a_0$ por el tamaño finito del conjunto. Aumentar el número de intervalos observados suele reducir las fluctuaciones estadísticas, aunque no cambia el modelo subyacente.

## 4. Reproducibilidad

La misma semilla, el mismo modelo y las mismas condiciones deben generar la misma trayectoria. Cambiar la semilla produce otra realización posible.

### Control de semillas aleatorias

Una semilla fija inicializa el generador de números aleatorios. Con el mismo código, versiones compatibles, parámetros y semilla se reproduce la misma secuencia de tiempos y eventos.

`run_ensemble` genera semillas hijas a partir de una semilla maestra, de modo que cada trayectoria del conjunto utiliza una corriente pseudoaleatoria independiente. Cambiar la semilla maestra produce otra realización del conjunto; no garantiza que cada valor individual sea distinto por una propiedad física, sino que cambia la secuencia pseudoaleatoria.

In [ ]:
first = simulator.run(motor, environment, t_max=0.5, seed=77)
repeat = simulator.run(motor, environment, t_max=0.5, seed=77)
other = simulator.run(motor, environment, t_max=0.5, seed=78)

print("Misma semilla: tiempos idénticos =", first.times_s == repeat.times_s)
print("Misma semilla: eventos idénticos =", first.events == repeat.events)
print("Semillas distintas: eventos idénticos =", first.events == other.events)

## 5. ATP consumido y pasos

El modelo registra una molécula de ATP en cada transición `atp_hydrolysis`. Cuenta también los eventos forward y backward. En esta red simplificada la hidrólisis y el paso no están estrictamente acoplados uno a uno: el desprendimiento puede ocurrir entre ambos. Esta limitación es una razón para mejorar posteriormente el ciclo mecanoquímico.

In [ ]:
records = []
for i, tr in enumerate(ensemble.trajectories):
    records.append({
        "trajectory": i,
        "atp_hydrolyzed": tr.atp_consumed_total,
        "forward_steps": tr.events.count("forward_step"),
        "backward_steps": tr.events.count("backward_step"),
        "detached": "detachment" in tr.events,
        "net_displacement_nm": tr.displacement_nm,
    })
summary_df = pd.DataFrame(records)
display(summary_df.head())
print("Fracción desprendida:", summary_df["detached"].mean())
print("ATP promedio registrado:", summary_df["atp_hydrolyzed"].mean())

### Interpretación del acoplamiento químico y mecánico

El contador de ATP aumenta cuando ocurre `atp_hydrolysis`. Los avances y retrocesos se cuentan por sus nombres de evento. En la red actual, el desprendimiento puede ocurrir desde `post_hydrolysis` antes de que se registre un paso; por eso una trayectoria puede consumir ATP sin completar el desplazamiento asociado al cierre del ciclo.

Esta separación es deliberada en el prototipo y señala una limitación del acoplamiento estequiométrico. Una versión mecanoquímica más completa deberá definir explícitamente las rutas de avance, retroceso y consumo químico, y comprobar su consistencia termodinámica.

## 6. Interpretación

La dispersión entre trayectorias es parte del resultado, no un error numérico. Para comparar con un experimento será necesario definir el observable de la misma manera, considerar trayectorias censuradas y utilizar parámetros cinéticos medidos o ajustados.

## Interpretación, límites y referencias

La variabilidad entre trayectorias es una predicción inherente al modelo estocástico, no un error numérico. Para comparar con un experimento deben coincidir la definición de velocidad, los criterios de inclusión/exclusión, el tratamiento de trayectorias censuradas, las condiciones de ATP y fuerza y los parámetros cinéticos.

Referencias:

- Gillespie, D. T. (1977). *Exact stochastic simulation of coupled chemical reactions*. The Journal of Physical Chemistry, 81(25), 2340–2361. [https://doi.org/10.1021/j100540a008](https://doi.org/10.1021/j100540a008)
- Ariga, T., Tomishige, M., & Mizuno, D. (2020). *Experimental and theoretical energetics of walking molecular motors under fluctuating environments*. Biophysical Reviews, 12, 503–510. [https://doi.org/10.1007/s12551-020-00684-7](https://doi.org/10.1007/s12551-020-00684-7)